# 0. 환경 설정 및 RAG 준비

- 코랩 환경에서 RAG 파이라인과 에이전트 구현을 위한 라이브러리 설치

## 0-1. 라이브러리 설치

- LangChain, Upstatge, ChromaDB 및 PDF 파서 설치
- 각 라이브러리에 대한 상세한 설명은 각 챕터에서 진행

In [ ]:
# RAG 파이프라인 및 LangChain 관련 라이브러리
# langchain-text-splitters: 문서를 의미 있는 단위(청크)로 분할하는 도구
!pip install langchain-text-splitters==0.3.9
# tiktoken: OpenAI 모델이 텍스트를 처리하는 단위인 '토큰'을 계산하는 라이브러리
!pip install tiktoken==0.11.0
# langchain-community: 다양한 외부 도구(Vector Store, Loader 등)와 연동하는 커뮤니티 제공 모듈
!pip install langchain-community==0.3.27
# langchain-openai: OpenAI 모델을 LangChain에서 사용하기 위한 모듈
!pip install langchain-openai==0.3.31
# langchain-upstage: Upstage 모델을 LangChain에서 사용하기 위한 모듈
!pip install langchain-upstage==0.7.3

# Vector Store (벡터 데이터베이스)
# chromadb: 텍스트 임베딩(벡터)을 저장하고 검색하는 경량 벡터 DB
!pip install chromadb==1.0.20

# Document Loaders (다양한 형식의 문서 로드용)
# pypdf, pymupdf, pypdfium2: PDF 파일에서 텍스트를 추출하기 위한 라이브러리들
!pip install pypdf==4.3.1
!pip install pymupdf==1.26.3
!pip install pypdfium2==4.3.0

## 0-2. Upstage API Key 설정

In [ ]:
# 구글 드라이브를 코랩 환경에 마운트.
# .env 파일과 같이 민감한 정보나 영구 저장할 파일을 관리하기 위함.
from google.colab import drive

drive.mount('/content/drive')

# API 키 파일(.env)이 저장된 기본 경로를 설정.
# 이 경로는 본인의 구글 드라이브 환경에 맞게 수정해야 함.
base_path = '/content/drive/MyDrive/Colab Notebooks/AI/09_RAG/'

In [ ]:
# Colab 환경에서 .env 파일을 생성하고 API 키를 저장하는 명령어.
# {your_api_key} 부분에 본인의 실제 Upstage API 키를 입력해야 함.
# '!'는 Colab에서 셸 명령어를 실행함을 의미.
!echo 'UPSTAGE_API_KEY={your_api_key}' > '{base_path}.env'

In [ ]:
# .env 파일에서 환경 변수를 로드하기 위한 라이브러리.
from dotenv import load_dotenv
# 운영체제(Colab 런타임)의 환경 변수를 가져오기 위한 함수.
from os import getenv

# .env 파일을 로드하여 환경 변수를 설정.
# 이 함수가 실행되면 .env 파일의 'KEY=VALUE'가 런타임의 환경 변수로 등록됨.
load_dotenv(base_path + '.env')

# getenv 함수를 사용해 'UPSTAGE_API_KEY'라는 이름의 환경 변수 값을 가져옴.
UPSTAGE_API_KEY = getenv('UPSTAGE_API_KEY')

# API 키가 성공적으로 로드되었는지 확인하고 메시지를 출력.
if UPSTAGE_API_KEY:
    print('Success API Key Setting!')
else:
    print(f'ERROR: Failed to load UPSTAGE_API_KEY from {base_path}')

## 0-3. 필요 개념 정리

### 0-3-1. 파싱과 청킹

1. **파싱 (Parsing)**
    - LLM은 기본적으로 `텍스트`만 이해할 수 있음.
    - PDF, HTML, DOCX 등 다양한 형식의 문서에는 텍스트 외에 레이아웃, 이미지, 서식 등 '노이즈'가 포함됨
    - **파싱**은 이러한 원본 문서에서 LLM이 이해할 수 있는 순수한 텍스트와 메타데이터(예: 출처, 페이지 번호)만 추출하는 과정임
    - RAG 파이프라인의 가장 첫 번째 'Ingestion(수집)' 단계에 해당
2. **청킹 (Chunking)**
    - LLM은 한 번에 처리할 수 있는 텍스트 양(Context Window)에 제한이 있음
    - RAG에서는 이보다 더 중요한 이유가 있음: **검색의 정확도.**
    - 만약 책 한 권을 통째로 하나의 '청크'로 만든다면, "배송비는 얼마인가?"라는 질문에 책 전체가 검색 결과로 나옴. 이는 LLM에게 너무 방대하고 희석된 정보를 제공하는 것.
    - 따라서, 문서를 검색에 용이하도록 **의미 있는** 작은 `청크` 단위로 나누는 작업이 필수적임. RAG는 '청크' 단위로 정보를 검색함.

### 0-3-2. 토크나이징과 청킹

- 두 개념은 자주 혼동되지만, 목적이 다름

1. **청킹**
    - **'검색(Retrieval)'**을 위한 텍스트 분할.
    - RAG 시스템이 정보를 검색하는 '단위'를 만드는 과정. (예: 200자짜리 문단)
    - `RecursiveCharacterTextSplitter` 등을 활용.
2. **토크나이징 (Tokenizing)**
    - **'LLM의 처리'**를 위한 텍스트 분할.
    - LLM이 텍스트를 이해하는 최소 단위(숫자 시퀀스)로 쪼개는 과정.
    - (예: "text" -> 29, 49)
    - `tiktoken` 라이브러리는 주로 이 토큰 '수'를 계산하여 모델의 컨텍스트 창 제한을 넘지 않는지 확인하는 용도로 사용됨.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 예시 텍스트
text = """
AI 온라인 서점입니다. 고객님의 소중한 문의에 감사드립니다.
배송 정책에 대해 안내해 드리겠습니다.
일반 도서의 경우 오후 3시 이전 주문 시 당일 발송됩니다.
주말 및 공휴일은 배송이 어렵습니다.
제주 및 도서 산간 지역은 추가 배송비가 발생할 수 있습니다.
주문 번호 order-123의 배송 상태를 조회하시려면 마이페이지에서 확인 부탁드립니다.
"""

# 다양한 chunk_size와 chunk_overlap으로 비교
# overlap: 청크 간 중복되는 부분의 크기.
# (예: 1번 청크의 끝부분과 2번 청크의 시작 부분이 겹치게 하여 문맥이 끊기는 것을 방지)
chunking_configs = [
    {'chunk_size': 30, 'chunk_overlap': 20},
    {'chunk_size': 100, 'chunk_overlap': 20},
]

for config in chunking_configs:
    chunk_size = config['chunk_size']
    chunk_overlap = config['chunk_overlap']

    # RecursiveCharacterTextSplitter 인스턴스 생성
    # 역할: 텍스트를 지정된 크기와 중복으로 청크로 분할
    # 'Recursive'의 의미: 문맥 유지를 위해 '\n\n', '\n', ' ' 등 구분자를 재귀적으로 시도함.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,  # 청크의 최대 크기
        chunk_overlap=chunk_overlap,  # 청크 간 겹치는 크기
        length_function=len,  # 길이를 문자 수(len)로 계산
        is_separator_regex=False,
    )

    # 텍스트를 청크(Document 객체 리스트)로 분할
    chunks = text_splitter.create_documents([text])

    print(f'\n[Chunk Size: {chunk_size}, Chunk Overlap: {chunk_overlap}]')
    print(f'생성된 청크 개수: {len(chunks)}')
    for i, chunk in enumerate(chunks):
        print(f'- 청크 {i+1} (길이: {len(chunk.page_content)}):')
        print(f"  '{chunk.page_content}'")

In [ ]:
import tiktoken

# 토크나이징 (tiktoken 사용) - 청킹과는 다름!
# 모델에 따라 인코딩(토큰화 방식)이 다름.
# 'cl100k_base'는 GPT-4, GPT-3.5-turbo, Upstage 모델 등에서 사용되는 인코딩 방식.
encoding = tiktoken.get_encoding('cl100k_base')

# .encode(): 텍스트를 토큰(숫자) 리스트로 변환
tokens = encoding.encode(text)

print(f'원문 텍스트 길이: {len(text)} 문자')
print(f'토큰 수: {len(tokens)} 토큰')
print(f'토큰 예시: {tokens[:10]}')  # 첫 10개 토큰 예시


### 0-3-3. 데이터 로드 및 분할

1. **데이터 로더 (Document Loaders)**
    - 다양한 형식(PDF, TXT, HTML 등)의 문서를 LangChain 표준 형식인 Document 객체로 불러오는 도구
    - `Document` 객체는 `page_content` (텍스트 내용)와 `metadata` (출처 등 부가정보)로 구성됨
    - `TextLoader`: `.txt` 파일 로드. 파일 하나당 `Document` 1개 생성
    - `PyMuPDFLoader`: `.pdf` 파일 로드. 페이지 하나당 `Document` 1개 생성
2. **텍스트 분할 (Text Splitter)**
    - `RecursiveCharacterTextSplitter`: `Document` 객체 리스트를 입력받아, 더 작은 `Document` 객체(청크) 리스트로 분할
    - `chunk_size`: 청크의 최대 크기. (RAG에서는 토큰이 아닌 문자 수 기준을 자주 사용)
        - (Trade-off) 너무 크면: 검색은 되지만 LLM에게 불필요한 정보가 많아져 답변 품질 저하.
        - (Trade-off) 너무 작으면: 문맥이 잘려나가 정보가 손실되고 검색 품질 저하.
    - `chunk_overlap`: 인접한 청크 간에 겹치는 문자 수. 청크 경계에서 문맥이 끊어지는 것을 방지

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. TextLoader를 사용하여 파일 불러오기
loader = TextLoader(base_path + 'shipping_policy.txt')
# .load(): 파일을 읽어 Document 객체 리스트를 반환 (TextLoader는 리스트에 1개)
documents = loader.load()

print(f'문서 개수: {len(documents)}')
if documents:
    # Document 객체의 텍스트 내용 확인
    print(f'내용 일부: {documents[0].page_content[:100]}')
    # Document 객체의 메타데이터(출처 파일 경로 등) 확인
    print(f'메타데이터: {documents[0].metadata}')

In [ ]:
# 2. RecursiveCharacterTextSplitter를 사용하여 문서 분할
# 실제 적용 시에는 데이터 특성에 맞게 chunk_size, chunk_overlap을 조정해야 함
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 청크 최대 크기 (문자 단위)
    chunk_overlap=20,  # 청크 간 겹치는 부분 (문자 단위)
    length_function=len,
    is_separator_regex=False,
)

# .split_documents(): Document 리스트를 받아 청크(Document) 리스트로 반환
# 원본 Document의 메타데이터는 생성된 청크들에 복사/상속됨
chunks = text_splitter.split_documents(documents)

print(f'생성된 청크 개수: {len(chunks)}')
for i, chunk in enumerate(chunks):
    print(f'\n청크 {i+1} (길이: {len(chunk.page_content)}):')
    print(f"{chunk.page_content}")
    # 청크에도 원본 파일 경로(source) 메타데이터가 포함된 것을 확인
    print(f'  메타데이터: {chunk.metadata}')

-------------

# 1. LangChain

- LLM, 프롬프트, 외부 도구(검색, DB, API) 등을 연결하여 강력한 애플리케이션(파이프라인)을 구축하는 프레임워크.

- LLM을 단순한 챗봇이 아닌, '엔진'으로 사용할 수 있게 하는 '오케스트레이션' 도구.

## 1-1. LLM Chain이란?

- LLM을 기반으로 출력을 생성하는 여러 구성 요소의 결합.
- **기본 구성**
    1. Prompt (사용자 입력을 받아 LLM에게 전달할 지시사항 생성)
    2. LLM (모델, 지시사항을 받아 텍스트 생성) 
    3. Output Parser (모델의 텍스트 출력을 원하는 형식(JSON, str 등)으로 변환).

## 1-2. LangChain이란?

- 위에서 언급한 LLM Chain을 쉽게 구축하도록 돕는 프레임 워크
- **기본 구성**
    1. `langchain-core`: LangChain의 핵심 문법(LCEL)과 기본 인터페이스 정의.
    2. `langchain-community`: Vector Store, Document Loader 등 다양한 외부 도구 연동.
    3. `langchain-openai`, `langchain-upstage` 등: 특정 LLM을 쉽게 사용하도록 돕는 파트너 패키지.
    4. `LangSmith`: LLM 체인의 실행 과정을 추적, 디버깅, 평가하는 플랫폼.

## 1-3. LCEL (LangChain Expression Language)

- LangChain의 핵심 문법으로, 파이프 (`|`) 연산자를 사용해 체인을 구성함.
- LCEL의 장점:
    1. **간결성 (Declarative):** 절차적 코드(예: `result1 = prompt.format(...)`, `result2 = model.invoke(result1)`) 대신, `chain = prompt | model | parser`와 같이 '흐름'을 선언적으로 정의.
    2. **스트리밍 및 비동기 지원:** LCEL로 정의된 체인은 별도 수정 없이 `.stream()`, `.ainvoke()` (비동기) 호출을 즉시 지원. 이는 절차적 코드로 구현하기 매우 복잡함.
    3. **조합성 (Composability):** 모든 체인은 'Runnable'이라는 단일 인터페이스를 따르므로, 작은 체인을 만들어 더 큰 체인의 부품처럼 조합할 수 있음.

### 1-3-1. Runnable 인터페이스 기반

- LCEL 체인을 구성하는 모든 요소(프롬프트, 모델, 출력 파서, 리트리버, 사용자 정의 함수 등)는 Runnable 인터페이스를 구현함
- Runnable 객체들은 파이프 (`|`) 연산자를 사용하여 연결됨
- **한 요소의 출력이 다음 요소의 입력으로 자동으로 전달**됨

### 1-3-2. 기본 사용 예시
- [Langchain Upstage 공식 문서](https://python.langchain.com/docs/integrations/providers/upstage/)
- [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html)

In [ ]:
# 예시 코드
from langchain.prompts import ChatPromptTemplate
from langchain_upstage import ChatUpstage
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt (Runnable): 입력(dict)을 받아 PromptValue 객체를 출력
# from_template: 템플릿 문자열을 사용하여 프롬프트 생성
prompt = ChatPromptTemplate.from_template('{topic}에 대해 설명해주세요')

# 2. Model (Runnable): PromptValue를 받아 AIMessage 객체를 출력
# ChatUpstage: Upstage의 챗 모델 인터페이스
# (API 키는 .env를 통해 로드된 환경 변수에서 자동으로 읽어 감)
model = ChatUpstage()

# 3. Parser (Runnable): AIMessage를 받아 문자열(str)을 출력
# StrOutputParser: 모델의 출력(AIMessage 객체)에서 텍스트 내용만 추출하여 단순 문자열로 반환
parser = StrOutputParser()

# LCEL을 사용한 체인 구성: prompt의 출력이 model로, model의 출력이 parser로 전달됨
chain = prompt | model | parser

# .invoke(): 체인을 실행하는 표준 메소드
# 입력값 {'topic': 'SSAFY'}는 첫 번째 Runnable(prompt)의 입력으로 들어감.
result = chain.invoke({'topic': 'SSAFY'})

print(result)

## 1-4. LangChain 활용

### 1-4-1. 답변 방식 제어 (Structured Output))

- LLM은 기본적으로 '텍스트'를 생성하므로, JSON이나 특정 형식의 출력을 강제하기 어려움.
- LangChain은 `Pydantic`과 `OutputParser`를 결합하여 LLM이 구조화된 데이터(JSON)를 반환하도록 유도함.

1. `Pydantic`: 데이터 유효성 검사 및 설정 관리 라이브러리.
    - **Python의 표준 타입 힌트**를 사용해, 원하는 데이터의 '스키마'를 클래스로 정의함.
    - 이 스키마 정의(특히 `Field`의 `description`)는 LLM에게 어떤 정보를 추출해야 하는지 알려주는 '힌트'가 됨
2. `JsonOutputParser`: Pydantic 모델과 연결되어 두 가지 핵심 작업을 수행함
    - **(입력 단계):** `get_format_instructions()`: Pydantic 모델을 기반으로 LLM에게 JSON 형식을 지시하는 프롬프트(텍스트)를 생성.
    - **(출력 단계):** LLM이 생성한 텍스트(JSON 문자열)를 파싱하여 Pydantic 객체(Python 객체)로 변환하고 유효성을 검사함.

In [ ]:
# 예시: 답변을 제목과 내용으로 구조화
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser  # 또는 PydanticOutputParser

# 1. Pydantic을 사용하여 원하는 출력 스키마 정의
class Summary(BaseModel):
    # Field의 'description'은 LLM에게 이 필드의 의미를 알려주는 중요한 힌트.
    title: str = Field(description='배송 방식')
    description: str = Field(description='배송 방식에 대한 정리')

# 데이터 불러오기
loader = TextLoader(base_path + 'shipping_policy.txt')
documents = loader.load()

# 3. ChatPromptTemplate 작성
prompt = ChatPromptTemplate.from_template(
    """
        당신은 어려운 문장을 쉽게 풀어서 설명하는 전문가입니다.
        1. 파일이 주어지면, 이 내용을 읽고 이해합니다. 답변에 추가하지 않습니다.
        2. 만약, 서로 다른 주제가 있다면 구분하여 다른 객체에 답변을 작성합니다.
        3. 당일, 익일 등은 오늘 오전과 같이 쉽게 풀어서 설명합니다.
        단, 지정된 JSON 형식으로 출력하세요:

{text}

{format_instructions}
    """
)

model = ChatUpstage()

# 2. JSONOutputParser를 Pydantic 모델과 연결
# pydantic_object 매개변수에 정의한 Summary 모델을 전달
parser = JsonOutputParser(pydantic_object=Summary)

# get_format_instructions(): Pydantic 모델을 기반으로 JSON 스키마 지시사항(텍스트) 생성
# 이 지시사항이 프롬프트의 {format_instructions} 부분에 주입됨.
format_instructions = parser.get_format_instructions()

# 4. 체인 구성 (Prompt -> Model -> Parser)
structured_chain = prompt | model | parser

# 5. 답변 확인
# .invoke()의 입력은 첫 번째 Runnable(prompt)의 입력 요구사항을 따름.
result = structured_chain.invoke(
    {'text': documents, 'format_instructions': format_instructions}
)

In [ ]:
# pprint (pretty-print): 딕셔너리나 JSON 같은 복잡한 자료구조를 읽기 쉽게 출력
from pprint import pprint

pprint(result)

### 1-4-2. 연쇄 요청

- LCEL의 장점인 '조합성(Composability)'을 활용한 예시.

- 1차 요청(정리)의 결과를 2차 요청(번역)의 입력으로 사용하는, 두 단계의 체인을 구현.

- `structured_chain` (앞서 만든 체인) 자체를 더 큰 `final_chain`의 '부품'으로 사용함.


1. **1단계 체인**: `structured_chain` (텍스트 -> JSON 요약)
2. **2단계 체인**: `translation_chain` (JSON 요약 -> 영어 번역)
3. **연결**: 두 체인을 파이프로 연결. 단, 1단계의 출력과 2단계의 입력 형식을 맞춰주기 위해 `lambda` 함수(RunnableLambda)를 중간에 사용

In [ ]:
# 2단계 체인(번역)을 위한 프롬프트
second_prompt = ChatPromptTemplate.from_template(
    """
        주어진 내용을 영어로 번역하세요:
        {json_content}
        응답 형식은 주어진 형식과 완전히 동일하게 반환하세요.
    """
)

# 2단계 체인(번역)을 위한 파서 (단순 문자열 출력)
second_parser = StrOutputParser()

# 2단계 체인 구성
translation_chain = second_prompt | model | second_parser

# 1단계 체인과 2단계 체인을 연결하여 최종 체인 구성
final_chain = (
    structured_chain  # 1. 입력(text, format_instructions) -> 1차 결과(JSON 객체)
    # 2. (RunnableLambda) 1차 결과를 2차 체인의 입력 형식에 맞게 변환.
    # 1차 결과(x)를 {'json_content': x} 딕셔너리로 래핑함.
    | (lambda x: {'json_content': x})
    | translation_chain  # 3. 래핑된 딕셔너리 -> 2차 결과(번역된 문자열)
)

# 최종 체인 실행 (입력은 1단계 체인의 요구사항과 동일)
final_result = final_chain.invoke(
    {'text': documents, 'format_instructions': format_instructions}
)

pprint(final_result)

### 1-4-3. 참고 (더 안정적인 스키마 정의)

- 현재 프롬프트는 `Summary` 객체를 단일 객체로 정의했음.

- 하지만 프롬프트 지시사항에는 "서로 다른 주제가 있다면 구분하여 다른 객체에 답변을 작성"하라고 요청함. (즉, 여러 개의 `Summary`가 생성될 수 있음을 암시)

- 이 경우, LLM이 JSON '리스트' `[Summary, Summary, ...]`를 반환하려 시도할 수 있으며, `JsonOutputParser(pydantic_object=Summary)`는 단일 객체만 기대하므로 파싱 오류가 발생할 수 있음.

- **해결책:** LLM의 출력과 Pydantic 스키마가 기대하는 바를 일치시켜야 함. 애초에 `List[Summary]`를 포함하는 래퍼(Wrapper) 클래스를 정의하는 것이 훨씬 안정적임.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from typing import List

# 1. 단일 객체 정의 (기존과 동일)
class Summary(BaseModel):
    title: str = Field(description='배송 방식')
    description: str = Field(description='배송 방식에 대한 정리')

# 2. (추천) 리스트를 포함하는 래퍼(Wrapper) 클래스 정의
# LLM이 생성할 JSON이 {"policies": [..., ...]} 구조를 갖도록 유도
class PolicyResponse(BaseModel):
    policies: List[Summary] = Field(description='배송 정책 목록')

# 3. 파서가 래퍼 클래스를 기대하도록 수정
# 이제 파서는 단일 객체가 아닌 PolicyResponse 객체를 기대함 (즉, 리스트를 포함한 객체)
stable_parser = JsonOutputParser(pydantic_object=PolicyResponse)

# 이렇게 수정하면 프롬프트의 지시사항과 파서의 기대치가 일치하여 안정성이 높아짐.

In [ ]:
# 모델 정의
model = ChatUpstage()

# **중요**: 1-4-3에서 정의한 'stable_parser'의 포맷 지시사항을 가져옵니다.
stable_format_instructions = stable_parser.get_format_instructions()

# 1-4-3의 파서로 체인 구성
# 1-4-1에서 사용한 prompt 활용
stable_structured_chain = prompt | model | stable_parser

# 체인 실행
print('--- 1-4-3의 안정적인 파서로 실행 중 ---')
result = stable_structured_chain.invoke(
    {'text': documents, 'format_instructions': stable_format_instructions}
)

# 결과 출력
# 1-4-1의 결과(단일 딕셔너리)와 달리, 
# policies라는 키(key) 아래에 Summary 객체들이 리스트 형태로 담긴 딕셔너리가 출력될 것
pprint(result)